In [58]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
import torch
from torchvision import transforms
from PIL import Image
import cv2
import numpy as np
import time
from sklearn.utils import resample

In [59]:
# CONSTANT
CLOCK_RAW_BASE_DIR = ""
COUNTER_RAW_BASE_DIR = ""
# merged datasets
OUTPUT_BASE_DIR = ""
CIRCUIT_LIST = [
    "simple_circuit",
    "montreal_circuit",
    "montmelo_circuit",
    "nurburgring_circuit"
]

In [60]:
def categorize_w(w):
    if w < -1.0:
        return "sharp_right"
    elif w < -0.05:
        return "right"
    elif w <= 0.05:
        return "straight"
    elif w <= 1.0:
        return "left"
    else:
        return "sharp_left"

In [62]:
TARGET_COUNT = 3000
VERSION = 'v3.3';
EXPORT_CSV_FILE_NAME = f'balanced_train_{VERSION}.csv'

### BALANCED DATASETS

In [46]:
df_balanced = []
balanced_classes = []


for base_dir in [CLOCK_RAW_BASE_DIR,COUNTER_RAW_BASE_DIR]:
    for circuit in CIRCUIT_LIST:

        # loading dataset
        df = pd.read_csv(os.path.join(base_dir,circuit,f"{circuit}-labels.csv"))
        
        df['w_category'] = df['w'].apply(categorize_w)
    
        # Process each class
        for category in df['w_category'].unique():
            class_df = df[df['w_category'] == category]
        
            if len(class_df) > TARGET_COUNT:
                # Downsample
                class_balanced = class_df.sample(n=TARGET_COUNT, random_state=42)
            else:
                # Oversample with replacement
                class_balanced = resample(class_df, replace=True, n_samples=TARGET_COUNT, random_state=42)
        
            balanced_classes.append(class_balanced)


df_balanced = pd.concat(balanced_classes).sample(frac=1, random_state=42).reset_index(drop=True)
print(df_balanced.describe())
df_balanced.to_csv(os.path.join(OUTPUT_BASE_DIR,EXPORT_CSV_FILE_NAME),index=False)

                   v              w
count  120000.000000  120000.000000
mean        4.573151      -0.004598
std         0.966082       0.930983
min         4.000000      -2.500000
25%         4.000000      -0.491920
50%         4.000000       0.001650
75%         4.946000       0.471480
max         7.000000       2.500000


## coppy all the image in temp image folder

In [54]:
import shutil

In [57]:
start_time = time.time()
total_images = 0

# create dir if not exist
os.makedirs(os.path.join(OUTPUT_BASE_DIR,'tmp_images'),exist_ok=True)

for base_dir in [CLOCK_RAW_BASE_DIR,COUNTER_RAW_BASE_DIR]:
    for circuit in CIRCUIT_LIST:
        image_count = 0
        src_dir = os.path.join(base_dir,circuit,'images')
        for filename in os.listdir(src_dir):
            if filename.lower().endswith(('.png')):
                image_count += 1
                full_file_name = os.path.join(src_dir, filename)
                shutil.copy(full_file_name, os.path.join(OUTPUT_BASE_DIR,'tmp_images'))
        total_images += image_count



# end time
end_time = time.time()
elapsed = end_time - start_time
print(f"Elapsed time: {elapsed:.2f} seconds")
print(f"{total_images} copied")

FileExistsError: [Errno 17] File exists: '/home/codezerro/Desktop/all_customs_datasets/v3/combine/datasets/balanced_train_v3.3.csv'

## proccess images

In [82]:
df = pd.read_csv(os.path.join(OUTPUT_BASE_DIR,EXPORT_CSV_FILE_NAME))

In [83]:
# start time
start_time = time.time()
error_images = 0
complete_images = 0
IMG_HEIGHT = 240
proccessed_image_dir = os.path.join(OUTPUT_BASE_DIR,'images')

# make dir for processed images
os.makedirs(proccessed_image_dir,exist_ok=True)


for image_name in df['image']:
    # 1. check image already proccessed or not
    full_path = os.path.join(os.path.join(proccessed_image_dir, image_name))
    if os.path.exists(full_path):
        continue

    # if proccessed is not exist on the images folder
    # GET THE FULL IMAGE INPUT PATH IF CURRENT `circuit`
    image_path = os.path.join(OUTPUT_BASE_DIR,'tmp_images',image_name)

    # Transform to tensor
    to_tensor = transforms.ToTensor()  # Converts PIL to [C,H,W] FloatTensor in [0,1]
    to_pil = transforms.ToPILImage()   # For visualization or debug


    try:
        # Load image with PIL and convert to RGB
        img = Image.open(image_path).convert('RGB')

        # To tensor and move to GPU
        tensor = to_tensor(img).unsqueeze(0).cuda()  # Shape: [1, 3, H, W]

        # Crop last 240 rows
        cropped_tensor = tensor[:, :, -IMG_HEIGHT:, :]  # Shape: [1, 3, 240, W]

        # Move to CPU and convert to numpy image (H, W, C) in uint8
        cropped_img_np = (cropped_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)

        # Save using OpenCV
        filename = os.path.basename(image_path)
        save_path = os.path.join(proccessed_image_dir,filename)
            
        cv2.imwrite(save_path, cv2.cvtColor(cropped_img_np, cv2.COLOR_RGB2BGR))

        complete_images += 1
    except Exception as e:
        print(f"Error processing {path}: {e}")
        error_images += 1
        
        
        
# end time
end_time = time.time()
elapsed = end_time - start_time

print("Total images {len(df['image'])}")
print(f"Complete to process {complete_images} images.")
print(f"Failed to process {error_images} images.")
print(f"Elapsed time: {elapsed:.2f} seconds")

Total images {len(df['image'])}
Complete to process 51958 images.
Failed to process 0 images.
Elapsed time: 504.35 seconds
